In [1]:
!pip install --prefer-binary "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps --prefer-binary xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sabxbf9f/unsloth_20ce3a6709964fa6ab434efa289f42a0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sabxbf9f/unsloth_20ce3a6709964fa6ab434efa289f42a0
  Resolved https://github.com/unslothai/unsloth.git to commit b3640802253f64117ee228718be7fab32e47aa5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.9 MB/s eta 0:00:0

In [2]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import os

# 2. LOAD MODEL (Start from the Base model to teach it the NEW CoD habit)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-bnb-4bit", # Or your Stage 2 model ID
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. LoRA ADAPTERS
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. DATA PREP (Using the COMPRESSED data)
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = [f"### Question:\n{i}\n\n### Reasoning:\n{o} " for i, o in zip(instructions, outputs)]
    return { "text" : texts, }

dataset = load_dataset("json", data_files="cod_train_data.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. TRAINING (Lower steps, shorthand is learned faster)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 4, # Higher batch size because sequences are shorter
        gradient_accumulation_steps = 2,
        max_steps = 40, # Shorthand habit forms quickly
        learning_rate = 1e-4, # Slightly lower LR for fine-tuning a style
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "outputs_cod",
        packing_strategy = "bfd",
    ),
)

trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 2 | Total steps = 40
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.083250
2,1.314442
3,1.133795
4,1.128644
5,1.127840
6,0.997608
7,0.897264
8,0.845274
9,0.777844
10,0.847207


Unsloth: Restored added_tokens_decoder metadata in outputs_cod/checkpoint-40/tokenizer_config.json.


TrainOutput(global_step=40, training_loss=0.5797424651682377, metrics={'train_runtime': 75.2949, 'train_samples_per_second': 4.25, 'train_steps_per_second': 0.531, 'total_flos': 662780388679680.0, 'train_loss': 0.5797424651682377, 'epoch': 1.6})

In [3]:
import time

def get_stats(model, tokenizer, question):
    prompt = f"### Question:\n{question}\n\n### Reasoning:\n"
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens = 512)
    end = time.time()

    full_text = tokenizer.decode(outputs[0])
    thought = full_text.split("<think>")[1].split("</think>")[0]

    latency = end - start
    tokens = len(tokenizer.encode(thought))
    return latency, tokens

# Test Question
q = "If a car travels 300 miles in 5 hours, and then increases speed by 10mph for 2 hours, total distance?"

lat, tox = get_stats(model, tokenizer, q)
print(f"COD MODEL -> Latency: {lat:.2f}s | Tokens: {tox}")
# Compare this against your Stage 2 model results

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


COD MODEL -> Latency: 39.59s | Tokens: 26


In [4]:
import time
import torch
import numpy as np
from unsloth import FastLanguageModel

def run_benchmark(model, tokenizer, questions):
    """
    Runs inference on a list of questions and records latency and token counts.
    """
    latencies = []
    token_counts = []

    # Set to inference mode
    FastLanguageModel.for_inference(model)

    print(f"Benchmarking {len(questions)} questions...")

    for q in questions:
        prompt = f"### Question:\n{q}\n\n### Reasoning:\n"
        inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

        # Warm-up (standard practice for accurate GPU benchmarking)
        _ = model.generate(**inputs, max_new_tokens = 1)

        # Actual measurement
        start_time = time.perf_counter() # Use perf_counter for high precision
        outputs = model.generate(
            **inputs,
            max_new_tokens = 512,
            use_cache = True,
            temperature = 0.1 # Keep it deterministic for benchmark
        )
        end_time = time.perf_counter()

        # Decode results
        full_text = tokenizer.batch_decode(outputs)[0]

        # Extract the thought block to count tokens
        try:
            thought_block = full_text.split("<think>")[1].split("</think>")[0]
            tokens = len(tokenizer.encode(thought_block))
        except IndexError:
            # Fallback if tags are missing
            tokens = 0

        latencies.append(end_time - start_time)
        token_counts.append(tokens)

    return {
        "avg_latency": np.mean(latencies),
        "avg_tokens": np.mean(token_counts),
        "tps": np.mean(token_counts) / np.mean(latencies) # Tokens per second
    }

# 1. Define Test Set (5 Representative Questions)
test_questions = [
    "A group of 4 people buy a $60 meal. If they leave a 15% tip, what is the total per person?",
    "If a train travels 240 miles in 4 hours, what is its speed in feet per second?",
    "Solve for x: 5x + 12 = 37. Show steps.",
    "If I have 3 dozen eggs and use 10 for a cake, how many are left?",
    "A rectangle has a length of 10 and a width of 5. If both increase by 20%, what is the new area?"
]

# 2. RUN BENCHMARK
# Note: You should run this once with the Stage-2 model loaded,
# then again with the Stage-4 (CoD) model loaded to get the comparison.
stats = run_benchmark(model, tokenizer, test_questions)

print("\n" + "="*30)
print("PRODUCTION EFFICIENCY REPORT")
print("="*30)
print(f"Avg Latency: {stats['avg_latency']:.3f} seconds")
print(f"Avg Thought Tokens: {stats['avg_tokens']:.1f} tokens")
print(f"Inference Speed: {stats['tps']:.2f} tokens/sec")
print("="*30)

Both `max_new_tokens` (=1) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Benchmarking 5 questions...


Both `max_new_tokens` (=1) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


PRODUCTION EFFICIENCY REPORT
Avg Latency: 37.977 seconds
Avg Thought Tokens: 126.0 tokens
Inference Speed: 3.32 tokens/sec


In [ ]:
from huggingface_hub import notebook_login
from unsloth import FastLanguageModel

# 1. Login to your account
# When the box appears, paste your "Write" token
notebook_login()

# 2. Define the Model IDs
# Replace 'your-username' with your actual Hugging Face username
hf_username = "nallaramu"
repo_name = "deliberate-qwen-2.5-3b-cod" # "cod" stands for Chain-of-Draft
model_id = f"{hf_username}/{repo_name}"

# 3. Save the model locally first (as a backup)
model.save_pretrained("deliberate_cod_adapter")
tokenizer.save_pretrained("deliberate_cod_adapter")

# 4. Push to Hugging Face Hub
# We include 'tags' to make your model searchable by recruiters
model.push_to_hub(
    model_id,
    tags = ["chain-of-draft", "reasoning", "efficiency", "unsloth", "distillation"],
    commit_message = "Initial release of Chain-of-Draft (CoD) optimized reasoning model"
)
tokenizer.push_to_hub(model_id)

print(f"🚀 Success! Your efficiency-optimized model is live at: https://huggingface.co/{model_id}")

In [14]:
import time
import torch
import pandas as pd
from unsloth import FastLanguageModel
from transformers import StoppingCriteria, StoppingCriteriaList

# 1. SETTINGS
MODEL_STAGING = "nallaramu/deliberate-qwen-2.5-3b-reasoning"
MODEL_COD = "nallaramu/deliberate-qwen-2.5-3b-cod"

test_questions = [
    "If a car travels 300 miles in 5 hours, and then increases speed by 10mph for 2 hours, what is the total distance traveled?",
    "A rectangular garden has a perimeter of 60 meters. If the length is twice the width, what is the area?",
    "A store offers a 20% discount on a $150 jacket. If sales tax is 8% after the discount, what is the final price?",
    "If 5 machines take 5 minutes to make 5 widgets, how long does it take 100 machines to make 100 widgets?",
    "If I have 12 apples and give half to a friend, then buy 4 more, how many do I have left?"
]

# 2. ROBUST TEXT-BASED STOPPING (The "Total Fix")
class RobustTextStopper(StoppingCriteria):
    def __init__(self, tokenizer, stop_words=["</answer>", "<|endoftext|>"]):
        self.tokenizer = tokenizer
        self.stop_words = stop_words

    def __call__(self, input_ids, scores, **kwargs):
        # Decode the last 15 tokens to see if any stop word has appeared ANYWHERE in them
        last_tokens_text = self.tokenizer.decode(input_ids[0][-15:])
        for stop_word in self.stop_words:
            if stop_word in last_tokens_text:
                return True
        return False

def run_benchmark(model_id, max_tokens=512):
    print(f"\n--- Loading {model_id} ---")
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_id, load_in_4bit=True)
    FastLanguageModel.for_inference(model)

    stop_criteria = StoppingCriteriaList([RobustTextStopper(tokenizer)])
    latencies, token_counts = [], []

    # 3. DEBUG PREVIEW
    print("Checking model output and forcing stop...")
    test_prompt = f"### Question:\n{test_questions[0]}\n\n### Reasoning:\n"
    test_inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

    # We use a smaller max_tokens for CoD because shorthand should NEVER need 512 tokens
    current_max = 100 if "cod" in model_id else 512

    # Run Benchmark
    for q in test_questions:
        prompt = f"### Question:\n{q}\n\n### Reasoning:\n"
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        torch.cuda.synchronize()
        start_time = time.time()

        outputs = model.generate(
            **inputs,
            max_new_tokens=current_max, # Hard limit: CoD doesn't need more than 100
            use_cache=True,
            stopping_criteria=stop_criteria,
            do_sample=False, # Greedy decoding for maximum speed
            pad_token_id=tokenizer.eos_token_id
        )

        torch.cuda.synchronize()
        end_time = time.time()

        latency = end_time - start_time
        prompt_len = inputs.input_ids.shape[1]
        new_tokens = outputs[0][prompt_len:]

        latencies.append(latency)
        token_counts.append(len(new_tokens))

    del model, tokenizer
    torch.cuda.empty_cache()
    return sum(latencies)/len(latencies), sum(token_counts)/len(token_counts)

# 4. EXECUTION
s_lat, s_tok = run_benchmark(MODEL_STAGING)
c_lat, c_tok = run_benchmark(MODEL_COD)

# 5. FINAL TABLE (With corrected math)
print("\n" + "="*50)
print("FINAL RECTIFIED BENCHMARK RESULTS")
print("="*50)

speedup = s_lat / c_lat
savings = ((s_tok - c_tok) / s_tok) * 100

results = {
    "Metric": ["Avg. Latency (Seconds)", "Avg. Total Tokens", "Inference Speedup", "Token Savings"],
    "Standard CoT": [f"{s_lat:.2f}s", f"{s_tok:.0f}", "-", "-"],
    "Chain-of-Draft": [f"{c_lat:.2f}s", f"{c_tok:.0f}", f"{speedup:.1f}x faster", f"{savings:.1f}% less"]
}

df = pd.DataFrame(results)
print(df.to_markdown(index=False))


--- Loading nallaramu/deliberate-qwen-2.5-3b-reasoning ---
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Checking model output and forcing stop...


...

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Checking model output and forcing stop...


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


FINAL RECTIFIED BENCHMARK RESULTS
| Metric                 | Standard CoT   | Chain-of-Draft   |
|:-----------------------|:---------------|:-----------------|
| Avg. Latency (Seconds) | 17.36s         | 4.53s            |
| Avg. Total Tokens      | 234            | 60               |
| Inference Speedup      | -              | 3.8x faster      |
| Token Savings          | -              | 74.3% less       |


In [15]:
# 1. Define the Save Path
save_directory = "deliberate-qwen-2.5-3b-cod-adapter"

# 2. Save the Adapter and Tokenizer
# This saves only the LoRA weights (~100MB), not the whole 6GB model
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model saved to {save_directory}")

# 3. Zip the folder for easy download
import shutil
shutil.make_archive(save_directory, 'zip', save_directory)

print(f"Archive created: {save_directory}.zip")

# 4. Trigger the download to your local machine
from google.colab import files
files.download(f"{save_directory}.zip")

Unsloth: Restored added_tokens_decoder metadata in deliberate-qwen-2.5-3b-cod-adapter/tokenizer_config.json.


Model saved to deliberate-qwen-2.5-3b-cod-adapter
Archive created: deliberate-qwen-2.5-3b-cod-adapter.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>